# Closed and open next-nearest-neighbour analyses

The analysis notebook for this
folder's data. Load every simulation in `../data` once — the plain `WASEP_*`
runs produced with `MODEL="WASEP"`, the `KLS_*` runs built from
$(\epsilon,\delta)$, and the `NNN_*` runs given explicit class weights — then
analyse the closed periodic and open-boundary cases in two independent
sections.

## Shared data loading

Every result is a CSV of recordings plus a JSON of metadata written by the same
engine, so one loader serves all three families. The rate description is part of
what identifies a run: two otherwise identical runs at different $\epsilon$ are
different simulations, and a run whose weights were given explicitly is kept
apart from the KLS run that happens to produce the same numbers.

In [1]:
import json
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_dir = Path.cwd()
if not (project_dir / 'data').is_dir() and (project_dir.parent / 'data').is_dir():
    project_dir = project_dir.parent
data_dir = project_dir / 'data'
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

In [2]:
# These identify a simulation. Final time is intentionally absent: a longer run
# of the same parameters is the same simulation, continued.
parameter_names = (
    'model', 'kls_epsilon', 'kls_delta', 'right_weights', 'left_weights',
    'length', 'walkers', 'field_E', 'bias_s', 'measurement_strength_k',
    'filling',
    'alpha_left_injection', 'gamma_left_removal',
    'delta_right_injection', 'beta_right_removal',
    'max_weight_ratio', 
    'record_interval', 'seed', 'initial_condition',
    'dynamics', 'boundary_condition',
)

float_columns = (
    'Time', 'CGFCloningNormalization',
    'MeanPopulationPotential', 'AcceptedCurrentRate',
    'ReferenceCurrentRate', 'PotentialSpread', 'EffectiveSampleSize',
)
integer_columns = (
    'ResamplingSteps', 'AcceptedWalkerEvents', 'RejectedWalkerEvents',
)
vector_columns = ('DensityProfile',)
required_columns = set(float_columns + integer_columns + vector_columns)

families = ('WASEP_', 'KLS_', 'NNN_')


def run_parameters(metadata, csv_path):
    """Return the identifying parameters, filled in for all three families."""
    # The filename family is the model: WASEP has no flanking classes at all,
    # KLS builds its eight rates from epsilon and delta, NNN was handed them.
    model = csv_path.name.split('_')[0]
    parameters = {name: metadata.get(name) for name in parameter_names}
    parameters['model'] = model
    # The WASEP engine calls its resampling threshold by its older name.
    if parameters['max_weight_ratio'] is None:
        parameters['max_weight_ratio'] = metadata.get('target_weight_factor')
    # Lists cannot go into a dictionary key; the ordering is (0,0) (0,1) (1,0) (1,1).
    for name in ('right_weights', 'left_weights'):
        if parameters[name] is not None:
            parameters[name] = tuple(parameters[name])
    return parameters


simulations = {}
csv_files = sorted(
    path for path in data_dir.rglob('*.csv')
    if path.name.startswith(families)
)
if not csv_files:
    raise FileNotFoundError(
        f'No {" / ".join(name + "*.csv" for name in families)} files found under {data_dir}'
    )

for csv_path in csv_files:
    metadata_path = csv_path.with_suffix('.json')
    if not metadata_path.exists():
        raise FileNotFoundError(f'Missing metadata file for {csv_path.name}')

    with metadata_path.open() as stream:
        metadata = json.load(stream)
    parameters = run_parameters(metadata, csv_path)
    key = tuple(parameters.items())  # Hashable dictionary key; no tmax.

    if key in simulations:
        raise ValueError(f'Duplicate parameter set: {csv_path}')

    frame = pd.read_csv(csv_path)
    if frame.empty:
        raise ValueError(f'{csv_path} contains no data rows')
    missing = required_columns.difference(frame.columns)
    if missing:
        raise ValueError(
            f'{csv_path.name} is missing columns: {sorted(missing)}'
        )

    # Convert every CSV column while loading: scalars become one-dimensional
    # arrays and JSON vectors become arrays indexed by [recording, component].
    # The frame itself is kept too, for ad-hoc work in the cells below.
    data = {
        name: frame[name].to_numpy(dtype=float)
        for name in float_columns
    }
    data.update({
        name: frame[name].to_numpy(dtype=np.int64)
        for name in integer_columns
    })
    data.update({
        name: np.asarray(frame[name].map(json.loads).tolist(), dtype=float)
        for name in vector_columns
    })

    simulations[key] = {
        'parameters': parameters,
        'tmax': metadata['tmax'],
        'metadata': metadata,
        'data': data,
        'frame': frame,
        'csv_path': csv_path,
        'metadata_path': metadata_path,
    }

print(f'Loaded {len(simulations)} simulation(s):')
for simulation in simulations.values():
    print(f"  {simulation['csv_path'].name}  (tmax={simulation['tmax']:g})")

Loaded 115 simulation(s):
  KLS_O_Direct_eps0.6_del0.0_L100_M2000_T20000.0_E6.5_s-2.515_k0.0_alpha1.0_gamma1.0_delta1.0_beta1.0_initalternating_seed12345.csv  (tmax=20000)
  KLS_O_Direct_eps0.6_del0.0_L100_M2000_T20000.0_E6.5_s-2.765_k0.0_alpha1.0_gamma1.0_delta1.0_beta1.0_initalternating_seed12345.csv  (tmax=20000)
  KLS_O_Direct_eps0.6_del0.0_L100_M2000_T20000.0_E6.5_s-3.015_k0.0_alpha1.0_gamma1.0_delta1.0_beta1.0_initalternating_seed12345.csv  (tmax=20000)
  KLS_O_Direct_eps0.6_del0.0_L100_M2000_T20000.0_E6.5_s-3.265_k0.0_alpha1.0_gamma1.0_delta1.0_beta1.0_initalternating_seed12345.csv  (tmax=20000)
  KLS_O_Direct_eps0.6_del0.0_L100_M2000_T20000.0_E6.5_s-3.515_k0.0_alpha1.0_gamma1.0_delta1.0_beta1.0_initalternating_seed12345.csv  (tmax=20000)
  KLS_O_Direct_eps0.6_del0.0_L100_M2000_T20000.0_E6.5_s-3.765_k0.0_alpha1.0_gamma1.0_delta1.0_beta1.0_initalternating_seed12345.csv  (tmax=20000)
  KLS_O_Direct_eps0.6_del0.0_L100_M2000_T20000.0_E6.5_s-4.015_k0.0_alpha1.0_gamma1.0_delta1.0_beta

In [3]:
def select_simulation(**wanted_parameters):
    """Select the unique dataset matching the supplied parameters."""
    unknown = set(wanted_parameters).difference(parameter_names)
    if unknown:
        raise KeyError(f'Unknown parameters: {sorted(unknown)}')

    matches = [
        simulation for simulation in simulations.values()
        if all(
            simulation['parameters'][name] == value
            for name, value in wanted_parameters.items()
        )
    ]
    if len(matches) != 1:
        names = '\n  '.join(sorted(s['csv_path'].name for s in matches))
        raise ValueError(
            f'Expected one match for {wanted_parameters}, found {len(matches)}'
            + (f':\n  {names}' if matches else '')
        )
    return matches[0]


def unpack(selected):
    """Return the arrays every section below plots, with shapes checked."""
    data = selected['data']
    expected_shapes = {
        'DensityProfile': (len(data['Time']), selected['parameters']['length']),
    }
    for name, expected_shape in expected_shapes.items():
        if data[name].shape != expected_shape:
            raise ValueError(
                f'{name} has shape {data[name].shape}, expected {expected_shape}'
            )
    return data


def describe_rates(selected):
    """Spell out the eight class rates, or say that there are none."""
    metadata = selected['metadata']
    if metadata.get('right_weights') is None:
        return 'uniform, one rate per direction (plain WASEP engine)'
    return (f"right {metadata['right_weights']}, left {metadata['left_weights']}"
            ' for flanking (a,d) = (0,0) (0,1) (1,0) (1,1)')


def rate_label(parameters):
    """One-line description of the hop rates, for plot titles."""
    if parameters['model'] == 'WASEP':
        return 'WASEP'
    if parameters['model'] == 'KLS':
        return (rf"KLS $\epsilon={parameters['kls_epsilon']:g}$, "
                rf"$\delta={parameters['kls_delta']:g}$")
    right = ', '.join(f'{w:g}' for w in parameters['right_weights'])
    return f'NNN rates ({right})'

# Results

Each section below takes one class of run from `../data` and compares it with
whatever is known independently: an exact formula where one exists, and
nothing at all where none does.

Throughout, the estimate quoted for the scaled cumulant generating function is
the **tail average** of the cloning normalisation, taken over the second half of
each run, and the counting field is referred to its own $s=0$ run at the same
field:

$$\chi(s)\;=\;\overline{g^{(Z)}(s)}\;-\;\overline{g^{(Z)}(0)},\qquad
g^{(Z)}(s)=\frac{\log Z_s(t)}{t}.$$

The subtraction matters: the engine reports a growth rate, not a CGF, and only
the difference is free of the reference weight.

In [ ]:
# Helpers shared by every results section below.  `simulations` was built by the
# loader above; these only ever read from it.

# Figures are written beside the notebook, split by what they are evidence
# for: `results/` holds the physics, `scalability/` holds the cost of running
# it.  Both are tracked, so a reader gets them without re-running anything.
FIGURES_DIR = project_dir / 'figures'
RESULTS_FIGURES = FIGURES_DIR / 'results'
SCALABILITY_FIGURES = FIGURES_DIR / 'scalability'
for _directory in (RESULTS_FIGURES, SCALABILITY_FIGURES):
    _directory.mkdir(parents=True, exist_ok=True)


def save_figure(figure, stem, into=None):
    """Write a figure as vector and raster, under the given section."""
    destination = into or RESULTS_FIGURES
    for suffix in ('pdf', 'png'):
        figure.savefig(destination / f'{stem}.{suffix}', dpi=200)
    print(f'saved figures/{destination.name}/{stem}.pdf and .png')


def runs_matching(**wanted):
    """Every loaded run whose parameters match, ordered by counting field."""
    unknown = set(wanted) - set(parameter_names)
    if unknown:
        raise KeyError(f'Unknown parameters: {sorted(unknown)}')
    found = [s for s in simulations.values()
             if all(s['parameters'][k] == v for k, v in wanted.items())]
    return sorted(found, key=lambda s: s['parameters']['bias_s'])


def growth_rate(selected):
    """The estimate of log Z_s(t)/t, read at the final recording.

    This column is already a cumulative time average, so it converges from
    below and its last row is the settled value.  Averaging it over a window
    would fold the early, unconverged rows back in and bias the result low --
    by about three per cent over the second half of these runs, which is the
    size of the effects being measured.
    """
    return float(selected['data']['CGFCloningNormalization'][-1])


def potential_rate(selected, fraction=0.5):
    """The second estimator, from the instantaneous population potential.

    Unlike the normalisation above this one *is* instantaneous, so here the
    time average is the right thing to take.  The two estimate the same growth
    rate by different routes, and their agreement is an internal check.
    """
    values = selected['data']['MeanPopulationPotential']
    return float(values[int(len(values) * fraction):].mean())


def cgf_of(selected, baselines):
    """Growth rate referred to the s = 0 run at the same field."""
    key = (selected['parameters']['field_E'],
           selected['parameters']['boundary_condition'],
           selected['parameters'].get('kls_epsilon'))
    return growth_rate(selected) - baselines[key]


def zero_bias_baselines(selection):
    """Map each field to the growth rate of its own unbiased run."""
    out = {}
    for s in selection:
        p = s['parameters']
        if abs(p['bias_s']) < 1e-12:
            out[(p['field_E'], p['boundary_condition'],
                 p.get('kls_epsilon'))] = growth_rate(s)
    return out


def biased(selection):
    """The runs of a selection that actually carry a counting field."""
    return [s for s in selection if abs(s['parameters']['bias_s']) > 1e-12]


def profiles(selected):
    """Density profile as an array indexed by [recording, site]."""
    return selected['data']['DensityProfile']


def missing(name):
    print(f'No runs found for: {name}.  Run the corresponding entry of the '
          f'results table into ../data first; this section is skipped.')


# Categorical hues, validated as a set for colour-vision deficiency.  Measured
# series wear them; a prediction is never a series -- it is drawn in ink, so
# that colour always means "this was simulated".
SERIES = ('#2a78d6', '#eb6834', '#1baf7a')
THEORY = '#0b0b0b'
INK, INK_SOFT, GRID_INK = '#0b0b0b', '#52514e', '#d9d8d4'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'font.size': 9, 'axes.titlesize': 9.5, 'axes.labelsize': 9,
    'legend.fontsize': 8.5, 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'axes.edgecolor': GRID_INK, 'axes.labelcolor': INK, 'text.color': INK,
    'xtick.color': INK_SOFT, 'ytick.color': INK_SOFT,
    'axes.grid': True, 'grid.color': GRID_INK, 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'legend.frameon': False,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02,
})


def frame_off(axes):
    for side in ('top', 'right'):
        axes.spines[side].set_visible(False)


def note_below(figure, text):
    figure.text(0.005, -0.06, text, ha='left', fontsize=8, color=INK_SOFT)

## 1. Does the estimate depend on the number of walkers?

The population-dynamics estimate is not guaranteed to have settled at any finite
walker number, and if it has not, a comparison against anything else is
measuring the shortfall rather than the physics. This section asks only whether
the estimate still moves as $M$ grows, which needs no external result: the open,
half-filled WASEP at $E=5$, $s=-5$, several seeds per walker number so that the
drift can be told from the scatter.

It does still move, and it moves in one direction. Note also how slowly: the
step from $M=16000$ to $M=64000$ is smaller than the step before it, but the
sequence has not flattened. Where an exact answer *is* available, at the small
lattices of `../validation`, the same estimator settles to within $0.02$ per cent
by $M=4000$ --- so what is left here is a property of the larger system and not
of the method.

In [ ]:
bias_runs = runs_matching(length=50, field_E=5.0, bias_s=-5.0,
                          boundary_condition='open', model='WASEP')
if not bias_runs:
    missing('walker-number dependence (L=50, E=5, s=-5)')
else:
    by_walkers = {}
    for run in bias_runs:
        by_walkers.setdefault(run['parameters']['walkers'], []).append(
            growth_rate(run))

    counts = np.array(sorted(by_walkers))
    means = np.array([np.mean(by_walkers[M]) for M in counts])
    errors = np.array([np.std(by_walkers[M], ddof=1) / np.sqrt(len(by_walkers[M]))
                       if len(by_walkers[M]) > 1 else 0.0 for M in counts])

    # With no external value to measure against, the largest population is the
    # best estimate available, and the others are read as displacements from it.
    reference = means[-1]
    print(pd.DataFrame({
        'M': counts, 'seeds': [len(by_walkers[M]) for M in counts],
        'CGF': means, 'std err': errors,
        'from largest M %': (means - reference) / abs(reference) * 100,
    }).to_string(index=False, float_format=lambda v: f'{v:.6f}'))

    figure, (left, right) = plt.subplots(1, 2, figsize=(9.2, 3.4))
    left.errorbar(counts, means, yerr=errors, color=SERIES[0], marker='o',
                  markersize=5.5, linewidth=2.0, capsize=3)
    left.set(xscale='log', xlabel='walkers $M$', ylabel=r'$\chi(s)$ per bond')

    displacement = np.abs(means[:-1] - reference)
    right.plot(counts[:-1], displacement, color=SERIES[0], marker='o',
               markersize=5.5, linewidth=2.0, label='measured')
    slope, offset = np.polyfit(np.log(counts[:-1]), np.log(displacement), 1)
    right.plot(counts[:-1], np.exp(offset) * counts[:-1] ** slope, color=THEORY,
               linestyle=':', linewidth=1.4, label=f'$\\propto M^{{{slope:.2f}}}$')
    right.plot(counts[:-1], displacement[0] * (counts[:-1] / counts[0]) ** -0.5,
               color=INK_SOFT, linestyle='--', linewidth=1.2,
               label=r'$M^{-1/2}$ for reference')
    right.set(xscale='log', yscale='log', xlabel='walkers $M$',
              ylabel=r'distance from the $M=%d$ estimate' % counts[-1])
    right.legend(loc='lower left')
    for ax in (left, right):
        frame_off(ax)
    figure.suptitle('The estimate has not settled in the walker number',
                    x=0.005, ha='left', fontsize=10.5, fontweight='bold')
    note_below(figure, f'Open WASEP, L = 50, E = 5, s = -5, t = 2000, '
                       f'{len(by_walkers[counts[0]])} seeds per point; error bars '
                       f'are the standard error of the mean over seeds.')
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    save_figure(figure, 'walker_number_dependence')
    plt.show()

## 2. Open SSEP against an exact result

With the field switched off and equal reservoirs, the open symmetric exclusion
process has a CGF known in closed form in the large-$L$ limit
(Derrida's matrix-ansatz solution).  With
$\rho_a=\alpha/(\alpha+\gamma)$, $\rho_b=\delta/(\delta+\beta)$ and

$$\omega(s)=\rho_a(e^{s}-1)+\rho_b(e^{-s}-1)+\rho_a\rho_b(e^{s}-1)(e^{-s}-1),$$

the leading behaviour is
$\psi_L(s)\simeq[\operatorname{arcsinh}\sqrt{\omega}]^2/(L-1)$ for
$\omega\ge0$, and $-[\arcsin\sqrt{-\omega}]^2/(L-1)$ below.

This is the sharpest check the repository has, because the formula carries no
statistical error.  It is run at two sizes on purpose: the formula is a
large-$L$ limit, so the pair separates the $1/L$ error of the *prediction* from
the bias of the *estimator*.

In [ ]:
def derrida_open_ssep_cgf(s, L, alpha, gamma, delta, beta):
    """Leading large-L open-SSEP CGF, in this code's exp(s Q) convention."""
    if alpha + gamma <= 0 or beta + delta <= 0:
        raise ValueError('Each boundary must have a positive total coupling')
    rho_a, rho_b = alpha / (alpha + gamma), delta / (beta + delta)
    exp_s = math.exp(s)
    omega = (rho_a * (exp_s - 1.0) + rho_b * (1.0 / exp_s - 1.0)
             + rho_a * rho_b * (exp_s - 1.0) * (1.0 / exp_s - 1.0))
    if omega >= -1e-14:
        value = math.asinh(math.sqrt(max(omega, 0.0))) ** 2
    elif omega >= -1.0 - 1e-14:
        value = -math.asin(math.sqrt(min(-omega, 1.0))) ** 2
    else:
        raise ValueError(f'omega={omega} lies outside the real formula domain')
    return value / (L - 1)


ssep_lengths = [L for L in (64, 100)
                if runs_matching(length=L, field_E=0.0, model='WASEP',
                                 boundary_condition='open')]
if not ssep_lengths:
    missing('open SSEP at E = 0')
else:
    figure, (left, right) = plt.subplots(1, 2, figsize=(9.2, 3.4))
    for colour, L in zip(SERIES, ssep_lengths):
        selection = runs_matching(length=L, field_E=0.0, model='WASEP',
                                  boundary_condition='open')
        baselines = zero_bias_baselines(selection)
        rows = []
        for run in biased(selection):
            p = run['parameters']
            rows.append((p['bias_s'], cgf_of(run, baselines),
                         derrida_open_ssep_cgf(
                             p['bias_s'], L, p['alpha_left_injection'],
                             p['gamma_left_removal'], p['delta_right_injection'],
                             p['beta_right_removal'])))
        table = pd.DataFrame(rows, columns=['s', 'measured', 'exact'])
        table['rel. error %'] = ((table['measured'] - table['exact'])
                                 / table['exact'].abs() * 100)
        print(f'\nL = {L}')
        print(table.to_string(index=False, float_format=lambda v: f'{v:.6f}'))

        left.plot(table['s'], table['measured'], color=colour, marker='o',
                  markersize=5.5, linestyle='none', label=f'simulated, L={L}')
        fine = np.linspace(table['s'].min(), table['s'].max(), 400)
        left.plot(fine, [derrida_open_ssep_cgf(s, L, 1, 1, 1, 1) for s in fine],
                  color=colour, linewidth=1.4, alpha=0.55)
        right.plot(table['s'], table['rel. error %'], color=colour, marker='o',
                   markersize=5.5, linewidth=2.0, label=f'L={L}')

    left.set(xlabel='counting field $s$', ylabel=r'$\chi(s)$ per bond')
    left.legend(loc='upper center')
    right.axhline(0.0, color=THEORY, linestyle='--', linewidth=1.2)
    right.set(xlabel='counting field $s$', ylabel='deviation from exact (%)')
    right.legend(loc='lower center')
    for ax in (left, right):
        frame_off(ax)
    figure.suptitle('Open SSEP against the exact large-$L$ CGF',
                    x=0.005, ha='left', fontsize=10.5, fontweight='bold')
    note_below(figure, 'E = 0, M = 2000, t = 2x10^4, equal reservoirs '
                       '(alpha = gamma = delta = beta = 1, so rho_a = rho_b = 1/2).  '
                       'Solid curves are the exact formula at the matching size.')
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    save_figure(figure, 'open_ssep_exact')
    plt.show()

## 3. Open WASEP under a driving field

With the field switched on the closed form of Section 2 no longer applies, and
the analytic saddle that would replace it is not distributed with this
repository. What follows is therefore the measurement on its own: the generating
function and the mean current against the counting field, and the recorded
density profile at three of those fields.

The profile is what carries the interest. Away from the ends of the range it
stops being flat, which is the lattice finding it cheaper to sustain an unusual
current with a structured density than with a uniform one.

In [ ]:
wasep_runs = runs_matching(length=100, field_E=5.0, model='WASEP',
                           boundary_condition='open')
if not wasep_runs:
    missing('open WASEP at E = 5, L = 100')
else:
    baselines = zero_bias_baselines(wasep_runs)
    rows = [{'s': run['parameters']['bias_s'],
             'CGF': cgf_of(run, baselines),
             '<J>': 99 * float(run['data']['AcceptedCurrentRate'][-1])}
            for run in biased(wasep_runs)]
    table = pd.DataFrame(rows).sort_values('s')
    print(table.to_string(index=False, float_format=lambda v: f'{v:.5f}'))

    figure, (left, right) = plt.subplots(1, 2, figsize=(9.2, 3.4))
    left.plot(table['s'], table['CGF'], color=SERIES[0], marker='o',
              markersize=5.5, linewidth=2.0)
    left.set(xlabel='counting field $s$', ylabel=r'$\chi(s)$ per bond')
    right.plot(table['s'], table['<J>'], color=SERIES[0], marker='o',
               markersize=5.5, linewidth=2.0)
    right.axhline(0.0, color=INK_SOFT, linewidth=0.9, linestyle=(0, (4, 3)))
    right.set(xlabel='counting field $s$', ylabel=r'mean current $\langle J\rangle$')
    for ax in (left, right):
        frame_off(ax)
    figure.suptitle('Open WASEP at $E=5$: generating function and current',
                    x=0.005, ha='left', fontsize=10.5, fontweight='bold')
    note_below(figure, 'L = 100, M = 2000, t = 2x10^4, half-filled reservoirs. '
                       'The current changes sign where the tilt cancels the drive.')
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    save_figure(figure, 'open_wasep_cgf_and_current')
    plt.show()

In [ ]:
# The recorded profile is the population average at the final time.
if wasep_runs:
    wanted = [-7.8, -5.4, -3.0]
    chosen = sorted([r for r in biased(wasep_runs)
                     if any(abs(r['parameters']['bias_s'] - w) < 1e-9
                            for w in wanted)],
                    key=lambda r: r['parameters']['bias_s'])
    if chosen:
        figure, grid = plt.subplots(1, len(chosen), figsize=(9.2, 2.9),
                                    sharey=True)
        grid = np.atleast_1d(grid)
        for ax, run in zip(grid, chosen):
            profile = profiles(run)[-1]
            ax.plot(np.arange(len(profile)), profile, color=SERIES[0],
                    linewidth=2.0)
            ax.axhline(0.5, color=INK_SOFT, linewidth=0.9,
                       linestyle=(0, (4, 3)))
            ax.set(title=f"$s = {run['parameters']['bias_s']:g}$", xlabel='site',
                   ylim=(-0.02, 1.02))
            frame_off(ax)
        grid[0].set_ylabel(r'density $\rho$')
        figure.suptitle('Recorded density profiles under the counting field',
                        x=0.005, ha='left', fontsize=10.5, fontweight='bold')
        note_below(figure, 'Open WASEP, E = 5, L = 100, M = 2000, population '
                           'average at the final time. The dashed line is the '
                           'reservoir density.')
        figure.tight_layout(rect=(0, 0, 1, 0.93))
        save_figure(figure, 'open_wasep_profiles')
        plt.show()

## 4. KLS: looking for a first-order dynamical phase transition

At $\epsilon=0.6$ and half-filled reservoirs, the additivity analysis predicts a
**first-order** transition in a window of the driving field.  The prediction is
recorded in `Simulator/docs/KLS_FIRST_ORDER_TARGETS.md`: a common-tangent
construction gives the counting field $s_c$ at which it sits, and a coexistence
interval $[J_1,J_2]$ across which the mean current should **jump**.

On the flat-profile branch the prediction is exactly Gaussian,
$\lambda_{\text{uni}}(s)=\tfrac{\sigma_*}{2}s(s+2E)$ with
$\sigma_*=8/15$, which gives a curve to compare against with no fitting.

The signature to look for is the jump in $\langle J\rangle$, not the value of
the CGF: section 1 showed the estimator's bias is an order of magnitude larger
than the convexification gap that distinguishes the transition, so absolute
values cannot settle the question but a discontinuity in shape can.

In [ ]:
KLS_SIGMA = 8.0 / 15.0
KLS_TARGETS = {6.5: dict(s_c=-3.765, J1=1.022, J2=1.459),
               7.0: dict(s_c=-3.351, J1=1.478, J2=1.945)}

kls_fields = [E for E in KLS_TARGETS
              if runs_matching(field_E=E, kls_epsilon=0.6,
                               boundary_condition='open', length=100)]
if not kls_fields:
    missing('KLS at epsilon = 0.6')
else:
    figure, grid = plt.subplots(2, len(kls_fields), figsize=(9.2, 6.0),
                                sharex='col', squeeze=False)
    for column, E in enumerate(sorted(kls_fields)):
        target = KLS_TARGETS[E]
        selection = runs_matching(field_E=E, kls_epsilon=0.6,
                                  boundary_condition='open', length=100)
        baselines = zero_bias_baselines(selection)
        rows = []
        for run in biased(selection):
            s = run['parameters']['bias_s']
            rows.append({
                's': s, 'CGF x (L-1)': cgf_of(run, baselines) * 99,
                'AP flat branch': KLS_SIGMA / 2 * s * (s + 2 * E),
                '<J>': 99 * float(run['data']['AcceptedCurrentRate'][-1]),
                'ESS/M': float(run['data']['EffectiveSampleSize'][
                    len(run['data']['Time']) // 2:].mean())
                    / run['parameters']['walkers'],
            })
        table = pd.DataFrame(rows).sort_values('s').reset_index(drop=True)
        table['d<J>/ds'] = table['<J>'].diff() / table['s'].diff()
        print(f'\nE = {E}   (predicted s_c = {target["s_c"]}, '
              f'jump {target["J1"]} -> {target["J2"]})')
        print(table.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

        upper, lower = grid[0][column], grid[1][column]
        upper.plot(table['s'], table['CGF x (L-1)'], color=SERIES[0],
                   marker='o', markersize=5.0, linewidth=2.0, label='simulated')
        upper.plot(table['s'], table['AP flat branch'], color=THEORY,
                   linestyle='--', linewidth=1.4, label='additivity, flat branch')
        upper.set(title=f'$E = {E:g}$', ylabel=r'$\chi(s)\,(L-1)$'
                  if column == 0 else None)

        lower.plot(table['s'], table['<J>'], color=SERIES[0], marker='o',
                   markersize=5.0, linewidth=2.0, label='simulated')
        # What a first-order transition would look like: a jump at s_c between
        # the two ends of the coexistence interval.
        lower.axvline(target['s_c'], color=THEORY, linewidth=1.2,
                      linestyle=(0, (4, 3)), label='predicted $s_c$')
        lower.plot([target['s_c'], target['s_c']], [target['J1'], target['J2']],
                   color=SERIES[1], linewidth=3.0, solid_capstyle='butt',
                   label='predicted jump')
        lower.set(xlabel='counting field $s$',
                  ylabel=r'$\langle J\rangle$' if column == 0 else None)
        for ax in (upper, lower):
            frame_off(ax)
        if column == 0:
            upper.legend(loc='lower right')
            lower.legend(loc='upper left')
    figure.suptitle('KLS at $\\epsilon = 0.6$: the predicted current jump is not seen',
                    x=0.005, ha='left', fontsize=10.5, fontweight='bold')
    note_below(figure, 'Open chain, L = 100, M = 2000, t = 2x10^4, half-filled '
                       'reservoirs, delta_KLS = 0.  Predictions from '
                       'Simulator/docs/KLS_FIRST_ORDER_TARGETS.md.')
    figure.tight_layout(rect=(0, 0, 1, 0.95))
    save_figure(figure, 'kls_first_order')
    plt.show()

## 5. Closed WASEP: a travelling density wave

On a ring nothing fixes where a structure sits, so the dynamics are invariant
under translation.  If the optimal history at strong bias is a moving density
profile, that invariance is broken by each realisation but restored by the
ensemble, and the question is whether the recorded profile shows it at all.

It does.  At every biased point below, the population-averaged profile carries a
large single-wavelength modulation, and its phase advances linearly in time.
The discriminator is exactly that phase drift: a **static** modulation and a
**travelling** one look identical in amplitude, and differ only in whether the
peak moves.

The one point that does not travel is $s=-E$.  There the tilt exactly cancels
the drive, the mean current vanishes, and the structure has nothing to advect
it: the modulation survives at full amplitude but stands still.

In [ ]:
ring_runs = runs_matching(boundary_condition='periodic', model='WASEP', length=100)
if not ring_runs:
    missing('closed WASEP on a ring')
else:
    rows = []
    for run in ring_runs:
        p = run['parameters']
        recorded = profiles(run)
        times = run['data']['Time']
        half = len(recorded) // 2
        late, late_t = recorded[half:], times[half:]
        L = late.shape[1]

        # First Fourier mode of the profile: amplitude says whether a single
        # long-wavelength structure is present, phase says whether it moves.
        coefficients = np.fft.rfft(late - late.mean(axis=1, keepdims=True), axis=1)[:, 1]
        amplitude = np.abs(coefficients).mean() / L
        peak_site = -np.unwrap(np.angle(coefficients)) * L / (2 * np.pi)
        drift, offset = np.polyfit(late_t, peak_site, 1)
        predicted = drift * late_t + offset
        spread = ((peak_site - peak_site.mean()) ** 2).sum()
        quality = 1 - ((peak_site - predicted) ** 2).sum() / max(spread, 1e-30)

        rows.append({'s': p['bias_s'], '<J>': L * float(run['data']['AcceptedCurrentRate'][-1]),
                     '|rho_1|': amplitude, 'drift (sites/1e3 t)': drift * 1000,
                     'linearity R^2': quality,
                     'ESS/M': float(run['data']['EffectiveSampleSize'][half:].mean())
                              / p['walkers']})
    table = pd.DataFrame(rows).sort_values('s').reset_index(drop=True)
    table['verdict'] = np.where(table['|rho_1|'] < 0.01, 'flat',
                        np.where((table['linearity R^2'] > 0.9)
                                 & (table['drift (sites/1e3 t)'].abs() > 1),
                                 'travelling', 'static'))
    print(table.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

    moving = table[table['verdict'] == 'travelling']
    figure, (left, right) = plt.subplots(1, 2, figsize=(9.2, 3.4))
    left.plot(table['s'], table['|rho_1|'], color=SERIES[0], marker='o',
              markersize=5.5, linewidth=2.0)
    left.set(xlabel='counting field $s$',
             ylabel=r'first-mode amplitude $|\hat\rho_1|$')
    left.annotate('flat at $s=0$', xy=(0.0, table['|rho_1|'].min()),
                  xytext=(-8, 18), textcoords='offset points', fontsize=8,
                  color=INK_SOFT, ha='right')

    right.plot(moving['<J>'], moving['drift (sites/1e3 t)'], color=SERIES[0],
               marker='o', markersize=5.5, linestyle='none', label='travelling')
    still = table[table['verdict'] == 'static']
    right.plot(still['<J>'], still['drift (sites/1e3 t)'], color=SERIES[1],
               marker='s', markersize=6.5, linestyle='none',
               label='static ($s=-E$, no current)')
    if len(moving) > 1:
        slope = np.polyfit(moving['<J>'], moving['drift (sites/1e3 t)'], 1)
        fine = np.linspace(0, moving['<J>'].max(), 50)
        right.plot(fine, np.polyval(slope, fine), color=THEORY, linestyle='--',
                   linewidth=1.3, label=f'linear, slope {slope[0]:.1f}')
    right.set(xlabel=r'mean current $\langle J\rangle$',
              ylabel='wave speed (sites per $10^3$ t)')
    right.legend(loc='upper left')
    for ax in (left, right):
        frame_off(ax)
    figure.suptitle('The ring modulation travels, at a speed set by the current',
                    x=0.005, ha='left', fontsize=10.5, fontweight='bold')
    note_below(figure, 'Closed WASEP, L = 100, M = 2000, filling 0.3, E = 10, '
                       't = 8x10^4.  Speed is the fitted drift of the first '
                       'Fourier mode over the second half of each run.')
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    save_figure(figure, 'ring_wave_speed')
    plt.show()

In [ ]:
# The same thing seen directly: density against site and time.  A vertical
# stripe is a standing modulation, a slanted one is a wave, and the slope of
# the slant is its speed.
if ring_runs:
    wanted = [-10.0, -8.0, -6.5]
    chosen = [r for r in ring_runs
              if any(abs(r['parameters']['bias_s'] - w) < 1e-9 for w in wanted)]
    if chosen:
        chosen.sort(key=lambda r: r['parameters']['bias_s'])
        figure, grid = plt.subplots(1, len(chosen), figsize=(9.6, 3.4), sharey=True)
        grid = np.atleast_1d(grid)
        for ax, run in zip(grid, chosen):
            recorded, times = profiles(run), run['data']['Time']
            window = (times >= 0.75 * times[-1]) & (times <= 0.85 * times[-1])
            image = ax.imshow(recorded[window], aspect='auto', origin='lower',
                              extent=(0, recorded.shape[1] - 1,
                                      times[window][0], times[window][-1]),
                              cmap='magma', vmin=0.0, vmax=1.0,
                              interpolation='nearest')
            ax.grid(False)
            ax.set(title=f"$s = {run['parameters']['bias_s']:g}$", xlabel='site')
        grid[0].set_ylabel('time')
        bar = figure.colorbar(image, ax=grid, fraction=0.02, pad=0.01)
        bar.set_label(r'density $\rho$')
        figure.suptitle('Ring density in space and time: standing at $s=-E$, '
                        'travelling beyond it',
                        x=0.005, ha='left', fontsize=10.5, fontweight='bold')
        note_below(figure, 'Closed WASEP, L = 100, M = 2000, filling 0.3, E = 10.  '
                           'A window of one tenth of the run is shown.')
        save_figure(figure, 'ring_space_time')
        plt.show()

# Scalability

The sections above read results the engine has already written.  This one
*produces* its own: it runs the engine over a grid of sizes and times it, so the
cost of the implementation can be read off rather than assumed.

Every point below is one **fixed task** — a run of stated model, lattice size,
population and final time, with the field, the bias and the seed held constant.
What is timed is the execution time of `run()`.  The throughput shown beside it
is that same run's accepted events divided by that same elapsed time, so the two
views never come from separate measurements.

Both are kept because they answer different questions:

- **Execution time** is what a run of a given specification costs, which is what you
  need in order to size a sweep.
- **Throughput** divides that cost by the work actually done.  It is what
  isolates the per-event claims of the notes: that the interacting update is
  $O(1)$ in the lattice size while the uniform one rebuilds $O(L/64)$ words per
  event, and that the heap pays $O(\log M)$ per event where the direct mechanism
  pays nothing.

The grids below take a few minutes in total.  They are deliberately modest;
widen `LENGTHS_MECHANISM` and `WALKERS_MECHANISM` to push further, at a cost
that grows roughly in proportion to the sizes added.

In [ ]:
import statistics
import time

# The engine sits beside the data this notebook already reads, not inside it.
if str(project_dir / 'Simulator') not in sys.path:
    sys.path.insert(0, str(project_dir / 'Simulator'))

from Python_Cpp_Interface import NNN

# Held fixed at every point, so that only the swept quantity varies.
BENCHMARK_COMMON = dict(E=5.0, s=-1.0, k=0.0, boundary='open',
                        alpha=1.0, gamma=1.0, delta=1.0, beta=1.0,
                        seed=12345, initial='alternating')
RATES = {
    'WASEP': dict(model='WASEP', epsilon=0.0, delta_kls=0.0),
    'KLS': dict(model='KLS', epsilon=0.6, delta_kls=0.0),
}
MECHANISMS = ('direct', 'heap', 'uniformized')

# The grids.  The model comparison reaches further in L than the mechanism one
# because it runs a single mechanism, and so costs a fraction as much.
BENCHMARK_TMAX = 50.0
LENGTHS_MODEL = (16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192)
LENGTHS_MECHANISM = (16, 32, 64, 128, 256, 512, 1024, 2048)
WALKERS_MECHANISM = (25, 50, 100, 200, 400, 800, 1600, 3200, 6400, 12800)
BENCHMARK_WALKERS = 200   # population held fixed while L is swept
BENCHMARK_LENGTH = 100    # lattice held fixed while M is swept


def time_run(repeats=1, **parameters):
    """Time one fixed task, taking the median of ``repeats`` runs."""
    seconds, events = [], None
    for _ in range(repeats):
        simulation = NNN(**parameters)
        start = time.perf_counter()
        simulation.run()
        seconds.append(time.perf_counter() - start)
        events = simulation.diagnostics()['accepted_walker_events']
    median = statistics.median(seconds)
    return {'seconds': median, 'events': events, 'rate': events / median}


def repeats_for(cost):
    """Repeat the cheap points; time the expensive ones once."""
    return 3 if cost <= 512 else 1


def processor_name():
    """Every number here is specific to the CPU, so the figures name it."""
    try:
        for line in Path('/proc/cpuinfo').read_text().splitlines():
            if line.startswith('model name'):
                return line.split(':', 1)[1].strip()
    except OSError:
        pass
    return 'this machine'


PROCESSOR = processor_name()
print(f'Benchmarking on {PROCESSOR}')

In [ ]:
# Two palettes, each validated as a set against colour-vision deficiency, and
# kept disjoint so that a colour never means a mechanism in one figure and a
# rate model in the next.  The green the notes use for `direct` sits about four
# units from their orange under protanopia, which is not separable; the green
# below is re-stepped to clear the floor while reading as the same hue.
MECHANISM_COLOUR = {'direct': '#1baf7a', 'heap': '#2a78d6',
                    'uniformized': '#eb6834'}
MODEL_COLOUR = {'WASEP': '#4a3aa7', 'KLS': '#e34948'}
INK, INK_SOFT, GRID_INK = '#0b0b0b', '#52514e', '#d9d8d4'
LINE_WIDTH, MARKER_SIZE = 2.0, 5.5

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'font.size': 9, 'axes.titlesize': 9.5, 'axes.labelsize': 9,
    'legend.fontsize': 8.5, 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'axes.edgecolor': GRID_INK, 'axes.labelcolor': INK, 'text.color': INK,
    'xtick.color': INK_SOFT, 'ytick.color': INK_SOFT,
    'axes.grid': True, 'grid.color': GRID_INK, 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'legend.frameon': False,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02,
})


def tidy(axes, xs, ticks):
    """Recede the frame; tick every point but label only a spaced subset."""
    for side in ('top', 'right'):
        axes.spines[side].set_visible(False)
    axes.set_xscale('log')
    axes.set_xticks(ticks)
    axes.set_xticks(xs, minor=True)
    axes.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))
    axes.xaxis.set_minor_formatter(plt.NullFormatter())


def spaced_ticks(values, every=2):
    """A subset of the swept values, far enough apart not to collide."""
    return sorted(set(list(values[::every]) + [values[-1]]))


def label_end(axes, x, y, text, colour):
    """Direct labels: with so few series, identity is never colour alone."""
    axes.annotate(text, xy=(x, y), xytext=(6, 0), textcoords='offset points',
                  color=colour, fontsize=8.5, va='center')


def caption(figure, text):
    figure.text(0.005, -0.06, text, ha='left', fontsize=8, color=INK_SOFT)


def save(figure, stem):
    save_figure(figure, stem, into=SCALABILITY_FIGURES)

## Cost against lattice size

Two sweeps over $L$, at fixed population and final time.  The first runs the
`direct` mechanism alone and compares the two rate models, which is where the
$O(1)$ local update meets the word-parallel rebuild.  The second runs all three
mechanisms, for each model separately.

In [ ]:
# One mechanism, both models, reaching as far in L as is affordable.
length_model_rows = []
for length in LENGTHS_MODEL:
    row = {'length': length}
    for name, rates in RATES.items():
        row[name] = time_run(
            repeats_for(length), L=length, M=BENCHMARK_WALKERS,
            tmax=BENCHMARK_TMAX, record_interval=BENCHMARK_TMAX,
            dynamics='direct', **BENCHMARK_COMMON, **rates,
        )
    length_model_rows.append(row)
    print(f"L={length:<6d} " + '  '.join(
        f"{n}: {row[n]['seconds']:7.3f} s ({row[n]['rate'] / 1e6:5.2f} Mev/s)"
        for n in RATES), flush=True)

In [ ]:
xs = [row['length'] for row in length_model_rows]
ticks = spaced_ticks(xs)
figure, (left, right) = plt.subplots(1, 2, figsize=(9.2, 3.5))

for name in RATES:
    colour = MODEL_COLOUR[name]
    seconds = [row[name]['seconds'] for row in length_model_rows]
    rates = [row[name]['rate'] / 1e6 for row in length_model_rows]
    left.plot(xs, seconds, color=colour, linewidth=LINE_WIDTH,
              marker='o', markersize=MARKER_SIZE, label=name)
    right.plot(xs, rates, color=colour, linewidth=LINE_WIDTH,
               marker='o', markersize=MARKER_SIZE, label=name)
    label_end(left, xs[-1], seconds[-1], name, colour)
    label_end(right, xs[-1], rates[-1], name, colour)

# Where one model's run becomes the more expensive of the two.
gaps = [row['WASEP']['seconds'] - row['KLS']['seconds']
        for row in length_model_rows]
crossover = None
for index in range(1, len(xs)):
    if gaps[index - 1] * gaps[index] < 0:
        low, high = math.log(xs[index - 1]), math.log(xs[index])
        crossover = math.exp(low + (high - low) * gaps[index - 1]
                             / (gaps[index - 1] - gaps[index]))
        break

left.set(yscale='log', xlabel='lattice size $L$',
         ylabel='execution time $t$ (seconds)')
right.set(xlabel='lattice size $L$', ylabel='throughput (M events/s)',
          ylim=(0, None))
for axes in (left, right):
    tidy(axes, xs, ticks)
    if crossover is not None:
        axes.axvline(crossover, color=INK_SOFT, linewidth=0.9,
                     linestyle=(0, (4, 3)), zorder=0)
if crossover is not None:
    left.annotate(f'crossover\n$L \\approx$ {crossover:.0f}',
                  xy=(crossover, left.get_ylim()[0]), xytext=(4, 6),
                  textcoords='offset points', fontsize=8, color=INK_SOFT,
                  va='bottom')
left.legend(loc='upper left')
figure.suptitle('Cost against lattice size: local update against word-parallel '
                'rebuild', x=0.005, ha='left', fontsize=10.5, fontweight='bold')
caption(figure, f'Direct mechanism, M = {BENCHMARK_WALKERS}, '
                f't = {BENCHMARK_TMAX:g}, open boundaries, E = 5, s = -1, '
                f'KLS at $\\epsilon$ = 0.6.  '
                f'Median of repeated runs on {PROCESSOR}.')
figure.tight_layout(rect=(0, 0, 0.97, 0.94))
save(figure, 'scaling_length_models')
plt.show()

In [ ]:
# All three mechanisms, each rate model measured separately.
length_mechanism_rows = {name: [] for name in RATES}
for name, rates in RATES.items():
    for length in LENGTHS_MECHANISM:
        row = {'length': length}
        for mechanism in MECHANISMS:
            row[mechanism] = time_run(
                repeats_for(length), L=length, M=BENCHMARK_WALKERS,
                tmax=BENCHMARK_TMAX, record_interval=BENCHMARK_TMAX,
                dynamics=mechanism, **BENCHMARK_COMMON, **rates,
            )
        length_mechanism_rows[name].append(row)
        print(f"{name:<6s} L={length:<6d} " + '  '.join(
            f"{m}: {row[m]['seconds']:7.3f} s" for m in MECHANISMS), flush=True)

In [ ]:
def mechanism_sweep(rows_by_model, key, xlabel, title, note, stem):
    # `title` is kept for the caller's documentation; the figure itself carries
    # no headline, so that it can be dropped straight under a LaTeX caption.
    """Three mechanisms against one swept size, a column per rate model.

    The upper row is the execution time of the fixed task.  The lower divides each
    mechanism by ``direct`` at the same point, which is the comparison the
    absolute times bury once both axes grow together.
    """
    figure, grid = plt.subplots(2, len(rows_by_model), figsize=(9.2, 5.2),
                                sharex='col')
    for column, (model, rows) in enumerate(rows_by_model.items()):
        xs = [row[key] for row in rows]
        ticks = spaced_ticks(xs)
        upper, lower = grid[0][column], grid[1][column]
        for mechanism in MECHANISMS:
            colour = MECHANISM_COLOUR[mechanism]
            seconds = [row[mechanism]['seconds'] for row in rows]
            ratio = [row[mechanism]['seconds'] / row['direct']['seconds']
                     for row in rows]
            upper.plot(xs, seconds, color=colour, linewidth=LINE_WIDTH,
                       marker='o', markersize=MARKER_SIZE, label=mechanism)
            lower.plot(xs, ratio, color=colour, linewidth=LINE_WIDTH,
                       marker='o', markersize=MARKER_SIZE, label=mechanism)
            label_end(lower, xs[-1], ratio[-1], f'{ratio[-1]:.1f}', colour)
        upper.set(yscale='log', title=model)
        lower.set(xlabel=xlabel, ylim=(0, None))
        lower.axhline(1.0, color=INK_SOFT, linewidth=0.8,
                      linestyle=(0, (4, 3)), zorder=0)
        if column == 0:
            upper.set_ylabel('execution time $t$ (seconds)')
            lower.set_ylabel(r'$t\,/\,t_{\mathrm{direct}}$')
            upper.legend(loc='upper left')
        for axes in (upper, lower):
            tidy(axes, xs, ticks)
    # No headline inside the image: the notebook heading above already names
    # it, and in the notes it would sit directly under the LaTeX caption.
    caption(figure, note)
    figure.tight_layout(rect=(0, 0, 0.97, 1.0))
    save(figure, stem)
    plt.show()


mechanism_sweep(
    length_mechanism_rows, 'length', 'lattice size $L$',
    'The three mechanisms against lattice size',
    f'M = {BENCHMARK_WALKERS}, t = {BENCHMARK_TMAX:g}, open boundaries, '
    f'E = 5, s = -1, KLS at $\\epsilon$ = 0.6.  '
    f'Median of repeated runs on {PROCESSOR}.',
    'scaling_length_mechanisms',
)

## Cost against population size

The same three mechanisms against $M$, at fixed lattice size, for each model
separately.  This is where the heap's $O(\log M)$ sift per event shows: on the
execution-time panel every mechanism grows in proportion to the population, and the
penalty separates only once each is divided by `direct`.

In [ ]:
walker_mechanism_rows = {name: [] for name in RATES}
for name, rates in RATES.items():
    for walkers in WALKERS_MECHANISM:
        row = {'walkers': walkers}
        for mechanism in MECHANISMS:
            row[mechanism] = time_run(
                repeats_for(walkers), L=BENCHMARK_LENGTH, M=walkers,
                tmax=BENCHMARK_TMAX, record_interval=BENCHMARK_TMAX,
                dynamics=mechanism, **BENCHMARK_COMMON, **rates,
            )
        walker_mechanism_rows[name].append(row)
        print(f"{name:<6s} M={walkers:<6d} " + '  '.join(
            f"{m}: {row[m]['seconds']:7.3f} s" for m in MECHANISMS), flush=True)

In [ ]:
mechanism_sweep(
    walker_mechanism_rows, 'walkers', 'population size $M$',
    'The three mechanisms against population size',
    f'L = {BENCHMARK_LENGTH}, t = {BENCHMARK_TMAX:g}, open boundaries, '
    f'E = 5, s = -1, KLS at $\\epsilon$ = 0.6.  '
    f'Median of repeated runs on {PROCESSOR}.',
    'scaling_walkers_mechanisms',
)

## One task, three mechanisms

The comparison at a single working size, as plain execution time.  This is the
measurement the *Measured cost* section of the notes tabulates; those numbers
were taken on a different machine, so read them against this figure rather than
transferring them.

In [ ]:
FIXED_TASK_TMAX = 200.0
fixed_task_rows = []
for name, rates in RATES.items():
    row = {'model': name}
    for mechanism in MECHANISMS:
        row[mechanism] = time_run(
            3, L=BENCHMARK_LENGTH, M=BENCHMARK_WALKERS, tmax=FIXED_TASK_TMAX,
            record_interval=FIXED_TASK_TMAX, dynamics=mechanism,
            **BENCHMARK_COMMON, **rates,
        )
    fixed_task_rows.append(row)
    print(f"{name:<6s} " + '  '.join(
        f"{m}: {row[m]['seconds']:6.3f} s" for m in MECHANISMS), flush=True)

figure, axes = plt.subplots(figsize=(6.4, 2.9))
height, gap = 0.24, 0.02
for group, row in enumerate(reversed(fixed_task_rows)):
    for slot, mechanism in enumerate(MECHANISMS):
        seconds = row[mechanism]['seconds']
        position = group + (1 - slot) * (height + gap)
        axes.barh(position, seconds, height=height,
                  color=MECHANISM_COLOUR[mechanism], zorder=2)
        # One palette step sits under 3:1 on white, so every bar is labelled.
        axes.annotate(f'{seconds:.2f} s', xy=(seconds, position),
                      xytext=(5, 0), textcoords='offset points',
                      va='center', fontsize=8.5, color=INK)

axes.set_yticks(range(len(fixed_task_rows)),
                [row['model'] for row in reversed(fixed_task_rows)])
axes.set_xlabel('execution time $t$ (seconds) - lower is better')
axes.set_xlim(0, max(row[m]['seconds'] for row in fixed_task_rows
                     for m in MECHANISMS) * 1.22)
axes.grid(axis='y', visible=False)
for side in ('top', 'right'):
    axes.spines[side].set_visible(False)
handles = [plt.Rectangle((0, 0), 1, 1, color=MECHANISM_COLOUR[m])
           for m in MECHANISMS]
axes.legend(handles, MECHANISMS, ncol=3, loc='lower right',
            bbox_to_anchor=(1.0, 1.0))
figure.suptitle('The same task under each event mechanism', x=0.005,
                ha='left', fontsize=10.5, fontweight='bold')
caption(figure, f'L = {BENCHMARK_LENGTH}, M = {BENCHMARK_WALKERS}, '
                f't = {FIXED_TASK_TMAX:g}, open boundaries, E = 5, s = -1, '
                f'KLS at $\\epsilon$ = 0.6.  '
                f'Median of three runs on {PROCESSOR}.')
figure.tight_layout(rect=(0, 0, 1, 0.92))
save(figure, 'scaling_mechanisms_fixed_task')
plt.show()